# Exercise 1. Prompting!
LLMs generate words (tokens) probabilistically in various ways, depending on the architecture and sampling method. As a consequence, the we words choose to provide the LLM impact what it generates. 

The process of "choosing the right words"  is called `prompt engineering`

:::{admonition} PAPER SPOTLIGHT: {cite:t}`hedderich-etal-2025-whats`
:class: fuchsia, dropdown
If you're interested in learning more about prompting, I suggest reading:   
[What’s the Difference? Supporting Users in Identifying the Effects of
Prompt and Model Changes Through Token Patterns](https://aclanthology.org/2025.acl-long.985.pdf)" by {cite:t}`hedderich-etal-2025-whats`

This paper presents "Spotlight", an approach to identify token patterns in prompts to make `prompt-engineering` more transparent for users. Paper was presented in at one of biggest NLP conferences `ACL2025`!
:::

## 1.1 Setup: Import Packages
If you have not already, please download the packages below (in venv or in UCloud) in your terminal:

```bash
pip install transformers torch
```

:::{admonition} Or download in notebook ... 
:class: tip, dropddown Remember, you can also download the packages in Jupyter notebooks with the %pip magic command as we have done in previous classes. 
:::

Import the packages

In [2]:
from transformers import AutoTokenizer
import transformers 
import torch 

## 1.2 Model Introduction
Let’s load Google’s `Flan-T5-base` and OpenAI’s `GPT-2`.

`Flan-T5-base` is an instruction-tuned version of Google’s influential [T5](https://huggingface.co/docs/transformers/en/model_doc/t5). It can follow prompts fairly well. However, as you will see, it does not behave quite like modern chatbots.


```{figure} ../figures/class6/flan-t5.png
---
name: flan-t5
width: 90%
---
Figure by {cite:t}`chung_scaling_2024`
```


`GPT-2`, in contrast, is **not** instruction-tuned, but it laid the foundation for ChatGPT and newer instruction-tuned LLMs.

Before continuing, take a moment to think about their underlying architectures, perhaps this will make it clear why `Flan-T5` is a bit special!

:::{admonition} QUESTION
:class: red
Which component(s) make up the architecture of these models? Decoder-only? Encoder-decoder? Encoder-only? 

Discuss with a friend and google it if you don't know, then check the answers below.

<details>
<summary>ANSWER</summary>
GPT2 is decoder-only! (<a href="https://jalammar.github.io/illustrated-gpt2/">See here</a>)     

Flan-T5 is encoder-decoder (<a href = "https://huggingface.co/docs/transformers/en/model_doc/t5">See here</a>)
</details>
:::

### Load Models
We'll use `transformers.pipeline` to load the models:

In [54]:
max_new_tokens = 100 # how many tokens max to generate

When loading `FLAN-T5`, we specify the task `text2text-generation`:

In [55]:
model = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model)
pipeline_t5 = transformers.pipeline(
    task = "text2text-generation",
    model=model,
    dtype=torch.float16, # a way to reduce memory
    max_new_tokens=max_new_tokens,
)

Device set to use mps:0


For `GPT-2`, the task is `text-generation`:

In [59]:
model = "openai-community/gpt2"

tokenizer = AutoTokenizer.from_pretrained(model)
pipeline_gpt = transformers.pipeline(
    task = "text-generation",
    model=model,
    dtype=torch.float16,
    max_new_tokens=max_new_tokens,
    return_full_text=False
)

Device set to use mps:0


:::{admonition} QUESTION
:class: red
Why does `Flan-T5` use `text2text-generation` while `GPT-2` uses `text-generation`?

<details>
<summary>ANSWER</summary>
Text-to-text generation, also called <code>sequence-to-sequence modeling</code>, maps an input sequence to an output sequence. It uses an <code>encoder–decoder</code> architecture, which processes information in both directions.

Text generation, on the other hand, generates text from left to right using a <code>decoder-only</code> architecture, as in GPT-2.
</details>
:::

## 1.3 Text Completion
Let's try to ask Flan-T5 a simple question:

In [14]:
pipeline_t5("What is the capital of Denmark?")

[{'generated_text': 'djurgrden'}]

The generated text above was obviously not what we wanted, let's phrase it in another way:

In [15]:
pipeline_t5("The capital of Denmark is")

[{'generated_text': 'Copenhagen'}]

:::{admonition} QUESTION
:class: red
Do you have any idea why the first phrasing, but the second one worked?

<details>
<summary>ANSWER</summary>
Flan-T5, like other language models, predicts the next word based on previous words. However, its sequence-to-sequence approach makes it more likely to perform better when completing an incomplete sentence than a sentence phrased as a question.
</details>
:::

What if we *really* want the question format? Try it yourself:
:::{admonition} HANDS-ON
:class: red
Create a better prompt!
* Write an instruction in the `task_prefix` below to make the prompt more complete to hopefully get a better answer.
* You can also experiment with the phrasing of the question, but keep it as a question!
* You may not be able to hit `Copenhagen`, but just focus on making it more meaningful than `djurgden`
:::


In [38]:
task_prefix = ""
prompt = f"{task_prefix}: What is the capital of Denmark?" # pass the prompt to pipeline_t5()

### 1.4 Summarization
A very useful application of LLMs is summarization! Let's say consider this student essay from [Class 1](/book/class1/001_simple_tokenization.ipynb):

In [51]:
student_text = """A matter of considerable controversy at present is the issue of whether distance-learning should be promoted as much as possible, or rather attending lectures in person should be allowed to take a predominant place in universities because this way of learning is superior than online degrees. From my perspective, online-teaching should be widely used among colleges and universities in terms of convenience and optimizing cost for educational institutions.
To begin with, distance-learning brings significant convenience for students in every corner of the world. In earlier times, students who were tired of commuting had to attend schools nearby, regardless of any differences in teaching facilities, teacher's qualifications or the school reputation. Now, however, students are able to apply for online-courses provided by top-of-the-range universities and colleges worldwide. 
Furthermore, the presence of video conferencing allows a teacher to teach a greater number of students. Consequently, educational institutions are able to optimize costs by increasing teacher-student ratios. Thanks to the economical online-teaching, universities and colleges are able to offer grants for students who have outstanding academic achievements but are unable to attend schools because of financial constrains.
Nevertheless, opponents of online-degrees would argue that attending lectures in person provides students an opportunity to communicate with teachers and other classmates. They further point out that traditional teaching approaches involving discussion and cooperation among students play a significant role in campus life. It is the real interactions and communications in class make education much more attractive.
By way of conclusion, it is my belief that distance-learning will become increasingly important in the future as the pace of life increases. However, discussions and interactions should be held via video conferencing frequently so that the joy of learning would not be diminished.
"""

:::{admonition} HANDS-ON
:class: red
- Create at least two different prompts called `prompt` (or `prompt1` / `prompt2`) for summarization by adding different `task_prefix` to `student_text` as done above.
- You can use the [f-string] formatting to that we did previously (see explanation below also!). 

Try it with both `GPT-2` and `Flan-T5`!

Note: You are also allowed to find a different text than `student_text` if you prefer that!
:::

:::{admonition} What is an f-string?
:class: red
It is very likely that you have seen the use of an f-string before this class! F-strings (formatted string literals) allow you to embed variables inside strings by putting an `f` before the opening quote of the string and using and curly brackets to include the variable:
```python
my_name = "Mina"
introduction = f"My name is {my_name}"

print(introduction) 
# outputs "My name is Mina"
```

Ypu can do this with as many variables you would like:
```python
my_name = "Mina"
my_city = "Aarhus" 
my_color = "green"
introduction = f"My name is {my_name} and I am from {my_city}. My favorite color is {my_color}."

print(introduction) 
# outputs "My name is Mina and I am from Aarhus. My favorite color is green."
```

As an alternative, you can also use the `format` method on a string:
```python
my_name = "Mina"
my_city = "Aarhus" 
my_color = "green"
introduction = "My name is {} and I am from {}. My favorite color is {}.".format(my_name, my_city, my_color)
# outputs "My name is Mina and I am from Aarhus. My favorite color is green."
```

You can also add two strings together:
```python
title = "The story of my life"
content = "I live in Aarhus, Denmark and have done so for many many years. I quite like it here. I also teach NLP which is SO meaningful to me!"

full_story = f"{title}: {content}"
print(full_story)
```
:::


Click to reveal solution:

In [71]:
# two prompts
task_prefix = "summarize this"
prompt = f"{task_prefix}: {student_text}"

t5 = pipeline_t5(prompt)
gpt = pipeline_gpt(prompt)

print("PROMPT TASK PREFIX:", task_prefix)
print("Flan T5 Summary:", t5[0]['generated_text'])
print("\n")
print("GPT-2 Summary:", gpt[0]['generated_text'])
print("\n")

# make a summary of this:
task_prefix = "write a summary of this"
prompt = f"{task_prefix}: {student_text}"

t5 = pipeline_t5(prompt)
gpt = pipeline_gpt(prompt)

print("PROMPT TASK PREFIX:", task_prefix)
print("Flan T5 Summary:", t5[0]['generated_text'])
print("\n")
print("GPT-2 Summary:", gpt[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


PROMPT TASK PREFIX: summarize this
Flan T5 Summary: Online-teaching should be widely used among colleges and universities in terms of convenience and optimizing cost for educational institutions.


GPT-2 Summary: This article was adapted from Peter Shropshire's essay "The Internet of Things", published in The British Journal of Computer Science, Vol. 52, No. 2, No. 1, May, 2010, pp. 1-5.




Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


PROMPT TASK PREFIX: write a summary of this
Flan T5 Summary: Online-teaching should be widely used among colleges and universities in terms of convenience and optimizing cost for educational institutions. However, opponents of online-teaching would argue that attending lectures in person provides students an opportunity to communicate with teachers and other classmates.


GPT-2 Summary: This article provides a brief overview of the latest developments in education in the U.S. as a whole. It shows that, at present, there is much uncertainty and uncertainty about the future. In addition, despite recent efforts to create a more efficient educational system, there are some areas where this could be improved. The first is the future of video conferences. In this article I have discussed the importance of these conferences in establishing a more integrated educational system. The second is the potential of video conferences to


:::{admonition} QUESTION
:class: red
How did this work? Was one better than the other? Did you experience that the models produce a new result everytime you run the chunk? Do you remember why that is?

<details>
<summary>ANSWER</summary>
Flan-T5 is seems better than GPT-2, but how you write the prompt also seems to affect performance.
<br><br>
The reason that models produce a new result every time is due to how their sampling parameters are configured! 
</details>
:::

## 1.5 Translation
Let's try another task, *translation*...